In [1]:
# Step 1: Clone the repo
!git clone https://github.com/Tusherbhomik/gorilla.git /kaggle/working/gorilla

Cloning into '/kaggle/working/gorilla'...
remote: Enumerating objects: 5409, done.
remote: Counting objects: 100% (14/14), done.
remote: Compressing objects: 100% (11/11), done.
remote: Total 5409 (delta 3), reused 6 (delta 3), pack-reused 5395 (from 1)
Receiving objects: 100% (5409/5409), 113.04 MiB | 32.65 MiB/s, done.
Resolving deltas: 100% (3960/3960), done.


In [2]:
# Step 2: Navigate to BFCL directory
%cd /kaggle/working/gorilla/berkeley-function-call-leaderboard

/kaggle/working/gorilla/berkeley-function-call-leaderboard


In [3]:
# Step 3: Install missing deps + bfcl (--no-deps avoids downgrading numpy/vllm)
!pip install -q typer tabulate overrides python-dotenv tenacity tqdm filelock networkx
!pip install -q -e . --no-deps
!pip uninstall -y flashinfer flashinfer-python 2>/dev/null || true
print("Done.")

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 1.9 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.4/295.4 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.6/108.6 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.3/31.3 MB 62.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 301.5/301.5 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 63.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 79.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 502.2/502.2 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.0/117.0 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━

In [4]:
# Step 4: Find python with vllm, patch base_oss_handler.py to use it
import subprocess, os, sys

handler_path = '/kaggle/working/gorilla/berkeley-function-call-leaderboard/bfcl_eval/model_handler/local_inference/base_oss_handler.py'

def has_vllm(py):
    try:
        r = subprocess.run([py, '-c', 'import vllm; print("ok")'],
                           capture_output=True, text=True, timeout=30)
        return r.returncode == 0 and 'ok' in r.stdout
    except (FileNotFoundError, subprocess.TimeoutExpired):
        return False

python_with_vllm = None
for py in ['python', 'python3', '/usr/local/bin/python3', '/usr/local/bin/python3.12',
           '/usr/bin/python3.12', sys.executable]:
    if has_vllm(py):
        python_with_vllm = py
        break

if not python_with_vllm:
    raise RuntimeError('No Python with vllm found. Run: !pip show vllm to debug.')

print(f'Python with vllm: {python_with_vllm}')

with open(handler_path, 'r') as f:
    content = f.read()

old = ('"vllm",\n'
       '                            "serve",\n'
       '                            str(self.model_path_or_id),')
new = (f'"{python_with_vllm}", "-m", "vllm.entrypoints.openai.api_server",\n'
       '                            "--model", str(self.model_path_or_id),')
content = content.replace(old, new)

if 'import sys' not in content.split('import subprocess')[0]:
    content = content.replace('import subprocess', 'import sys\nimport subprocess', 1)

with open(handler_path, 'w') as f:
    f.write(content)

os.environ['VLLM_ATTENTION_BACKEND'] = 'TRITON_ATTN'

assert 'vllm.entrypoints.openai.api_server' in open(handler_path).read(), 'Patch failed!'
print(f'Handler patched.')
print(f'Backend: {os.environ["VLLM_ATTENTION_BACKEND"]}')


Backend: TRITON_ATTN


In [5]:
# Step 5: Generate (inference) — env vars passed inline to guarantee subprocess inherits them
!VLLM_ATTENTION_BACKEND=TRITON_ATTN FLASHINFER_DISABLE_JIT=1 python -m bfcl_eval generate \
    --model tusherbhomik/qwen2.5-1.5b-hgr-5340-r2 \
    --test-category simple_python \
    --num-gpus 1 \
    --backend vllm \
    --allow-overwrite

Generating results for ['tusherbhomik/qwen2.5-1.5b-hgr-5340-r2']
Running full test cases for categories: ['simple_python'].
config.json: 1.40kB [00:00, 633kB/s]
tokenizer_config.json: 100%|███████████████████| 694/694 [00:00<00:00, 3.84MB/s]
tokenizer.json: 100%|██████████████████████| 11.4M/11.4M [00:00<00:00, 19.6MB/s]
chat_template.jinja: 2.51kB [00:00, 1.05MB/s]
Max context length: 32768
╭───────────────────── Traceback (most recent call last) ──────────────────────╮
│ /kaggle/working/gorilla/berkeley-function-call-leaderboard/bfcl_eval/__main_ │
│ _.py:191 in generate                                                         │
│                                                                              │
│   188 │   │   lora_modules=lora_modules,                                     │
│   189 │   )                                                                  │
│   190 │   load_dotenv(dotenv_path=DOTENV_PATH, verbose=True, override=True)  │
│ ❱ 191 │   generation_main(args)     

In [6]:
# Step 5: Evaluate
!python -m bfcl_eval evaluate \
    --model tusherbhomik/qwen2.5-1.5b-hgr-5340-r2 \
    --test-category simple_python

Number of models evaluated: 0it [00:00, ?it/s]
📈 Aggregating data to generate leaderboard score table...
🏁 Evaluation completed. See /kaggle/working/gorilla/berkeley-function-call-leaderboard/score/data_overall.csv for overall evaluation results on BFCL V4.
See /kaggle/working/gorilla/berkeley-function-call-leaderboard/score/data_live.csv, /kaggle/working/gorilla/berkeley-function-call-leaderboard/score/data_non_live.csv, /kaggle/working/gorilla/berkeley-function-call-leaderboard/score/data_multi_turn.csv, /kaggle/working/gorilla/berkeley-function-call-leaderboard/score/data_agentic.csv and /kaggle/working/gorilla/berkeley-function-call-leaderboard/score/data_format_sensitivity.csv for detailed evaluation results on each sub-section categories respectively.
